# Практическое долгосрочное задание 1
## 👄 Интерполяция формы губ по видео



## ЧАСТЬ 0. 🎯 Введение

Нам в руки попали **обрывки данных** — редкие снимки губ человека, произносящего звуки **восточноевропейского алфавита** 🕵️

Наш агент, внедрённый глубоко под прикрытием, снимал его **раз в секунду** — строго по **равномерной временно́й сетке**. Никаких отклонений, никаких дополнительных замеров. Только сухие факты: кадр, ещё кадр, и снова тишина.




Между этими кадрами — **пустота**. Звуки шли один за другим, но мы видим лишь **редкие вспышки мимики**: губы то округляются, то растягиваются, то сжимаются — и всё это **урывками**, без плавных переходов. Мимика кажется **рваной**, движения — **дёргаными**. Смотреть на такое больно 😖

---

### 🚀 Наша финальная задача

#### Научиться воспроизводить **гладкую анимацию** для… 🌀

…впрочем, узнаем **потом**. 😏

---

### 🎬 А пока — начнём с малого!

#### Попробуем восстановить хотя бы **отдельные звуки** 🗣️

<table>
<tr>
<td align="center">👄</td>
<td align="center">🎞️</td>
<td align="center">✨</td>
<td align="center">🔊</td>
</tr>
<tr>
<td align="center">губы</td>
<td align="center">кадры</td>
<td align="center">сгладим</td>
<td align="center">зазвучит!</td>
</tr>
</table>

---

> 💭 **Шаг за шагом:**
> 1. 📍 Соберём редкие кадры
> 2. 🧮 Построим интерполяцию
> 3. 🎥 Оживим анимацию
> 4. 🌍 А потом — уже что-то большее…

In [4]:
#Импорт нужных библиотек

from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np
import glob
from collections import defaultdict
import draw
from scipy.interpolate import barycentric_interpolate, CubicSpline, BSpline

import pickle

In [5]:
#Загрузка данных
with open('LipsArrayHalfTime.np' , 'rb') as f:
    LipsData = pickle.load(f)

### 📦 Структура данных: `LipsData`

`LipsData` — это **словарь**, ставящий в соответствие каждой букве её временну́ю траекторию движения губ:

$$
\text{LipsData} : \quad \text{буква} \;\longmapsto\; \mathbf{X} \in \mathbb{R}^{n \times 40 \times 2}
$$

---

#### 📐 Что означает каждый индекс

$$
\mathbf{X}[t,\, p,\, c], \qquad
\begin{aligned}
&t = 0, \dots, n-1 \\
&p = 0, \dots, 39 \\
&c \in \{0, 1\}
\end{aligned}
$$

| Индекс | Обозначение | Смысл |
|:---:|:---:|:---|
| $t$ | **время** | номер такта — дискретный отсчёт вдоль оси времени |
| $p$ | **точка** | 40 ключевых точек контура губ (20 внешних + 20 внутренних) |
| $c$ | **координата** | $c=0 \Rightarrow x$, $\;c=1 \Rightarrow y$ на плоскости |

---

#### 🕰️ О такте

> **Такт** — это условная единица времени.  
> Можно считать, что $1$ такт $\approx 0.1$ с, но эта привязка несущественна:  
> для интерполяции важна лишь **последовательность** отсчётов, а не их физическая длительность.

---

#### 🧠 Как это читать

Для фиксированной буквы $\ell$ имеем:

$$
\text{LipsData}[\ell] \;=\; \bigl\{\, \mathbf{x}_t \,\bigr\}_{t=0}^{\,n-1},
\qquad
\mathbf{x}_t \in \mathbb{R}^{40 \times 2}
$$

То есть каждая буква — это **растянутая во времени последовательность** «снимков» формы губ:

$$
\mathbf{x}_0 \;\longrightarrow\; \mathbf{x}_1 \;\longrightarrow\; \cdots \;\longrightarrow\; \mathbf{x}_{n-1}
$$

где каждый $\mathbf{x}_t$ — плоская конфигурация из 40 точек:

$$
\mathbf{x}_t \;=\;
\bigl\{\, (x_p,\, y_p) \,\bigr\}_{p=0}^{39}
$$


### 🎬 Визуализация: функция `animate_lips`

Преподаватель предоставил **готовую функцию** для интерактивной визуализации:

```python
animate_lips(sequence, interval=...)
```
Предлагается оставить значение интервала равным 200. Однако во время интерполяции желательно увеличивать это время пропорционально отнощению желанного количества отрисованных кадров к исходному количеству кадров
### Ниже представлен пример для буквы А

In [6]:
draw.animate_lips(LipsData['А'], interval=200)

## 📐 Часть 1. Интерполяционный многочлен Лагранжа


#### НАПОМНИМ

Пусть даны $n$ узлов $t_0 < t_1 < \dots < t_{n-1}$ и значения функции в них $y_0, y_1, \dots, y_{n-1}$.

**Задача:** найти многочлен $P(t)$ степени $\le n-1$, который **точно проходит через все узлы**:

$$
P(t_i) = y_i, \qquad i = 0, \dots, n-1
$$

---

#### 🧮 Формула Лагранжа

Решение единственно и записывается в виде

$$
P(t) \;=\; \sum_{i=0}^{n-1} y_i \, L_i(t)
$$

где **базисные многочлены Лагранжа**:

$$
L_i(t) \;=\; \prod_{\substack{j=0 \\ j \ne i}}^{n-1} \frac{t - t_j}{\,t_i - t_j\,}
$$

---

#### ✨ Ключевое свойство

$$
L_i(t_k) \;=\;
\begin{cases}
1, & i = k \\[4pt]
0, & i \ne k
\end{cases}
$$

То есть каждый базисный многочлен «отвечает» **только за свой узел**.  
Отсюда мгновенно следует:

$$
P(t_k) = \sum_i y_i \, L_i(t_k) = y_k \quad ✔
$$

---



#### 🤔 Что же делать в части 1 ???

Мы применим эту формулу к **каждой из 80 координат** наших губ:

$$
(x_0, y_0),\;(x_1, y_1),\;\dots,\;(x_{39}, y_{39})
$$

и восстановим **пропущенные кадры** между известными узлами.

#########################################################################################################################

### 🎯 Задание

Дан словарь `LipsData`, где:

- 🔑 ключ — буква,
- 📊 значение — массив `(n, 40, 2)`.

**Требуется:**

1. Для каждой буквы $\ell$ построить **интерполяционный многочлен Лагранжа**
   для каждой из $40 \times 2 = 80$ координат.

   
   В качестве узлов использовать **все исходные кадры**:

   $$
   t_i = 0,\; 1,\; 2,\; \dots,\; n-1
   $$

3. Вычислить значения многочлена на **вдвое более плотной сетке**:

   $$
   t = 0,\; 0.5,\; 1,\; 1.5,\; 2,\; 2.5,\; \dots,\; n-1
   $$

   То есть кадров станет **в 2 раза больше**, чем было изначально.

4. Собрать результат в массив формы `(2n − 1, 40, 2)`.

5. **Визуализировать** результат функцией `animate_lips`.

---

#### ✅ Что должно получиться

- 📈 Каждая буква — восстановленная последовательность `(n, 40, 2)`.
- 🎬 Плавная анимация, где промежуточные кадры заполнены.



In [ ]:
##TODO
##ЗДЕСЬ ДОЛЖЕН БЫТЬ ВАШ КОД
## Результат запишите в переменную Task1LispPoints, представляющую собой 
## словарь , где ключ - буква , а значение - массив размера (n, 40, 2) (значение 2n − 1 зависит от буквы)

Task1LispPoints = {}

for letter, data in LipsData.items():
    n = data.shape[0]
    t = np.arange(n)
    t_new = np.arange(0, n, 0.5)
    
    # Приводим к форме (n, 80) для векторизации по всем координатам сразу
    Y = data.reshape(n, 80)
    
    # Barycentric interpolation - это численно устойчивая реализация формулы Лагранжа
    # Применяем к каждой из 80 координат
    Y_new = np.array([barycentric_interpolate(t, Y[:, i], t_new) for i in range(80)]).T
    
    # Возвращаем исходную форму (2n-1, 40, 2)
    new_data = Y_new.reshape(len(t_new), 40, 2)
    Task1LispPoints[letter] = new_data

# Визуализация
draw.animate_lips(Task1LispPoints['В'], interval=100)

### ⚠️ Внимание!

> **Не ждите, что результаты будут прям красивыми** 😅  


> 🧠 **Если что-то не получается — это нормально!**  
> Подумайте, **почему** результат выглядит именно так,  
> и выпишите свои мысли в ячейке ниже. ✍️

#### ✍️ Запишите здесь, что вы наблюдаете и почему, по вашему мнению,
####    результат выглядит именно так.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
# При интерполяции Лагранжа на большом числе узлов наблюдается феномен Рунге.
# Губы неестественно дергаются и вылетают за пределы нормальной анатомии,
# потому что многочлен высокой степени пытается пройти ровно через каждую точку,
# усиливая шум исходных данных.

## 🌊 Часть 2. Кубические сплайны

Многочлен Лагранжа на большом числе узлов **нестабилен** — феномен Рунге делает своё дело 🔥

**Сплайны** решают эту проблему: вместо одного «здоровенного» многочлена мы строим **много маленьких кусочков**, которые аккуратно сшиваются между собой.

---

### 📐 Кубический сплайн дефекта 1

Дан отрезок $[a, b]$ и узлы $t_0 < t_1 < \dots < t_n$.  
Кубический сплайн $S(t)$ — это функция, которая:

1. На каждом подынтервале $[t_i, t_{i+1}]$ является **кубическим многочленом**:
   $$
   S(t) = a_i + b_i (t - t_i) + c_i (t - t_i)^2 + d_i (t - t_i)^3
   $$

2. **Проходит через все узлы**: $S(t_i) = y_i$

3. **Непрерывна** вместе с первой и второй производной:
   $$
   S, \; S', \; S'' \;\text{— непрерывны на всём } [a, b]
   $$

4. На концах задаётся **краевое условие** (например, $S''(a) = S''(b) = 0$ — *natural spline*).

> 💡 **Дефект 1** означает, что «склейка» идеальная: разрывается только третья производная, а $S$, $S'$, $S''$ — гладкие.

---

### 🧩 Задание 2.1 — заменить Лагранжа сплайнами

1. Реализуйте метод построения **кубического сплайна дефекта 1** из лекции.
2. Примените его к той же задаче, что и в Части 1:
   - узлы — все исходные такты $t_i = 0, 1, 2, \dots, n-1$,
   - значения — координаты губ,
   - восстановление на сетке с шагом $0.5$.



Проверка:

```python
animate_lips(SplineLipsLetters['А'], interval=100)
```

> 🧠 **Сравните** с результатом Лагранжа.  
> Запишите в ячейке ниже, **чем сплайны отличаются** от многочлена Лагранжа визуально.

---

## 🔗 Задание 2.2 — склеиваем звуки в слова

Теперь попробуем **соединять звуки** между собой — как в реальной речи, где один звук плавно перетекает в другой.

### 🛠️ Подготовка

1. **Вручную просмотрите** каждый звук через `animate_lips`.
2. **Обрежьте** крайние кадры, где рот закрывается или стоит без движения.  
   💡 *Можно использовать уже восстановленные кадры из задания 2.1 — они глаже.*
3. Оставьте только «рабочую» часть, где **губы действительно меняются**.

### 🧪 Эксперимент

Соберите **связки из 2–3 букв**, например:

```python
'АБ',  'БА',  'АБВ',  'ОБА',  ...
```

Для каждой связки:

- 🔴 **сначала** соедините куски **многочленом Лагранжа**,
- 🔵 **потом** — **кубическим сплайном**.
#### *Это независимые действия, то есть должно быть два результата: звук получен Лагранжем, а потом Сплайном*

Сравните анимации.

### 🎬 Финальный тест

Соберите слово:

```
абёба
```

и покажите его анимацию — сначала Лагранжем, потом сплайном.

---

## ✍️ Что записать в отчёте

Опишите **своими словами**:

- 🔍 Что происходит на **стыках** звуков у Лагранжа? А у сплайнов?
- 📉 Есть ли **разрывы**, **скачки**, **выбросы**?
- 👀 Какой метод выглядит **естественнее**?
- 🤔 Почему?

---

> 💡 **Подсказка:** чтобы соединить два звука, просто **склейте их массивы по оси времени**:
> ```python
> combined = np.concatenate([part1, part2], axis=0)
> ```
> А затем **интерполируйте уже склейку** — там будет видно, справляется ли метод с «переходом».

In [9]:
#TODO: Часть 2.1
Task2LispPoints = {}

for letter, data in LipsData.items():
    n = data.shape[0]
    t = np.arange(n)
    t_new = np.arange(0, n, 0.5)
    
    # CubicSpline нативно поддерживает многомерные данные через axis=0
    cs = CubicSpline(t, data, axis=0)
    new_data = cs(t_new)
    
    Task2LispPoints[letter] = new_data

# Визуализация
draw.animate_lips(Task2LispPoints['В'], interval=100)

In [ ]:
#TODO: Часть 2.2
def trim_edges(data, ratio=0.15):
    n = data.shape[0]
    trim_n = int(n * ratio)
    # Защита от слишком коротких последовательностей
    trim_n = min(trim_n, n // 3) 
    return data[trim_n : n - trim_n]

word_letters = ['А', 'Б', 'Ё', 'Б', 'А']
trimmed_parts = [trim_edges(Task2LispPoints[letter]) for letter in word_letters]

concatenated = np.concatenate(trimmed_parts, axis=0)

# Интерполируем уже склеенную последовательность для сглаживания стыков
n_concat = concatenated.shape[0]
t_concat = np.arange(n_concat)
t_concat_new = np.arange(0, n_concat, 0.5)

# Сплайн для слова
cs_word = CubicSpline(t_concat, concatenated, axis=0)
word_data_spline = cs_word(t_concat_new)

# Лагранж для слова 
Y_concat = concatenated.reshape(n_concat, 80)
Y_concat_new_lagrange = np.array([barycentric_interpolate(t_concat, Y_concat[:, i], t_concat_new) for i in range(80)]).T
word_data_lagrange = Y_concat_new_lagrange.reshape(len(t_concat_new), 40, 2)

print(f"Форма склеенного слова (сплайн): {word_data_spline.shape}")

# Визуализация: Сплайн
# draw.animate_lips(word_data_spline, interval=100)

# Визуализация: Лагранж
draw.animate_lips(word_data_lagrange, interval=100) 

Форма склеенного слова (сплайн): (148, 40, 2)


In [17]:
# При попытке интерполировать склеенное слово многочленом Лагранжа анимация
# превращается в точку. На стыках звуков Лагранж дает огромные выбросы, так как степень
# многочлена растет пропорционально длине слова, и феномен Рунге
# усиливается экспоненциально. Кубические сплайны, напротив, обеспечивают гладкое
# соединение. Локальность сплайнов означает, что добавление нового звука влияет
# только на соседние сегменты, поэтому анимация слова "абёба" выглядит плавно и
# естественно, без разрывов и скачков.

## 📉 Часть 3. Метод наименьших квадратов (МНК)

Всё, что мы делали до сих пор, было **интерполяцией** — многочлен **обязан** пройти через каждую точку.  
Именно это и приводит к **феномену Рунге**: пытаясь угодить всем узлам сразу, многочлен «сходит с ума» на краях.

**Метод наименьших квадратов** поступает иначе: мы больше **не требуем** точного прохождения через точки.  
Вместо этого ищем функцию, которая **в среднем** проходит как можно ближе ко всем данным — и это делает её куда **устойчивее** и **гладче**.

🎯 Мы попробуем сгладить наши губы МНК и сравнить результат с тем, что уже получили.


> #### 🚨 ВНИМАНИЕ! 🚨
>
> ##### 📉 Часть 3 (МНК) — последняя ОБЯЗАТЕЛЬНАЯ ЗАДАЧА для сдачи (и, естественно, допуска к экзамену)
>
> ✅ Дальше — только по желанию 🌟


### 🧱 Основа Части 3: смена фич

До сих пор функция губ **обязана была проходить через каждый кадр** — это интерполяция: Лагранжу за неё достаётся от феномена Рунге 🔥, а кубическим сплайнам из Части 2 — от дёрганости исходных данных.

**МНК** (лекция 5) меняет договор: функция проходит **мимо** точек, лишь бы сумма квадратов отклонений была минимальной. И на первый план выходит вопрос — **из каких функций составлять приближение**:

$$
g(t) \;\approx\; \sum_{k=0}^{K-1} \theta_k\, \varphi_k(t)
$$

- В лекции 5 фичами были степени $1,\; t,\; t^2,\; \dots$ — и мы видели, чем это грозит (матрица Гильберта).
- Теперь возьмём в качестве фич **базисные сплайны**: $\varphi_k(t) = B_k(t)$ — те самые $B$-сплайны из лекции 4. По теореме Карри — Шёнберга любой сплайн однозначно раскладывается по этому локальному базису.

Соберём **матрицу фич** — строка $i$ отвечает кадру $t_i$:

$$
G_{ik} = B_k(t_i), \qquad
\min_{\boldsymbol{\theta}} \lVert G\boldsymbol{\theta} - \mathbf{g} \rVert^2
\;\;\Longrightarrow\;\;
\hat{\boldsymbol{\theta}} = (G^{\mathsf{T}} G)^{-1}\, G^{\mathsf{T}} \mathbf{g}
$$

— система нормальных уравнений из лекции 5. Приятный нюанс: $G$ зависит только от временно́й сетки кадров и узлов базиса, но не от самих координат губ — значит, **одна и та же $G$ обслуживает все $80$ МНК-задач**. Соберём правые части в матрицу $Y$ размера $(n,\, 80)$ и решим всё разом:

$$
\hat{\Theta} = (G^{\mathsf{T}} G)^{-1}\, G^{\mathsf{T}} Y
$$

Предсказание на новых сетках: вычисляем **вектор базиса** на подкадре $t^*$ с компонентами $b_k(t^*) = B_k(t^*)$ и используем уже готовые коэффициенты:

$$
\hat{g}(t^*) \;=\; \mathbf{b}(t^*)^{\mathsf{T}}\, \hat{\boldsymbol{\theta}}
$$

> 💭 На исходных кадрах кривая пройдёт **рядом, но не точно** по точкам — это не баг, а суть МНК-сглаживания: оно «прощает» те рывки данных, которые интерполяция добросовестно воспроизводила.

### Кратные узлы в B-сплайнах

---

#### 1. Определение кратного узла

**Кратный узел** — это точка сетки сплайна, в которой **совпадают координаты нескольких узлов**. Вместо того чтобы разносить узлы последовательно по оси $x$ (например, $0, 1, 2, 3$), несколько узлов искусственно помещаются в одну и ту же координату (например, $0, 0, 0, 0, 1, 2...$). 

Количество совпадающих узлов в одной точке называется его **кратностью** ($k$). 

В случае **кубических B-сплайнов** (степень $p = 3$) кратность граничных узлов делают равной $k = p + 1 = \mathbf{4}$. Вектор узлов (knot vector) для отрезка $[a, b]$ принимает вид:
$$T = \{\underbrace{a, a, a, a}_{4 \text{ узла слева}}, \, t_1, \, t_2, \, \dots, \, t_m, \, \underbrace{b, b, b, b}_{4 \text{ узла справа}}\}$$

---

#### 2. Какую проблему это решает? (Проблема «вырождения» на краях)

Каждая базисная функция $B_{i,3}(x)$, вычисляемая по рекуррентной формуле Кокса — де Бора, обладает свойством **локального носителя**. Она отлична от нуля только на строго ограниченном участке из нескольких соседних узлов, а за его пределами равна нулю.

* **Если узлы равномерные (без кратности):** На границах отрезка ($x = a$ и $x = b$) «хвосты» крайних базисных функций будут плавно затухать. На самих краях абсолютно все базисные функции $B_{i,3}(x)$ автоматически **стремятся к нулю**.
* **Математический тупик:** Если все базисные функции на краю равны нулю, то любая их линейная комбинация $S(x) = \sum c_i B_{i,3}(x)$ в граничных точках гарантированно выдаст ноль ($S(a) = 0$ и $S(b) = 0$). Сплайн физически не сможет принимать ненулевые значения на краях.

---

#### 3. Зачем это нужно в Методе Наименьших Квадратов (МНК)?

Метод наименьших квадратов пытается подобрать коэффициенты $c_i$ так, чтобы сплайн максимально точно повторял форму аппроксимируемой функции $f(x)$. 

Если не использовать кратные узлы, МНК столкнется со следующим искажением:
1. **Насильственное притягивание к нулю:** Какую бы функцию вы ни приближали (например, константу $f(x) = 5$ или экспоненту), сплайн на краях отрезка будет противоестественно «падать» в ноль. Возникнет огромная погрешность аппроксимации на границах.
2. **Потеря управления:** Алгоритм МНК потеряет способность настраивать поведение сплайна в окрестности точек $a$ и $b$.

#### Как спасают ситуацию 4 кратных узла:
Когда мы зажимаем края, складывая 4 узла в одну точку, мы «сплющиваем» локальные носители крайних функций. Происходит математическое перераспределение весов по формуле Кокса — де Бора:
* В точке $x = a$ самая первая базисная функция становится строго равной единице: $B_{0,3}(a) = 1$.
* Все остальные базисные функции в этой точке зануляются: $B_{1,3}(a) = 0, B_{2,3}(a) = 0 \dots$

Благодаря этому значение сплайна на левой границе теперь полностью освобождено от зануления и определяется всего одним коэффициентом МНК:
$$S(a) = c_0 \cdot \underbrace{B_{0,3}(a)}_{1} + c_1 \cdot \underbrace{B_{1,3}(a)}_{0} + \dots = c_0$$

> Кратные узлы нужны **не для того, чтобы занулить функции, а чтобы спасти сплайн от автоматического зануления**. Они дают алгоритму МНК свободу «поднять» или «опустить» края сплайна до реальных значений аппроксимируемой функции $f(x)$ за счет свободных коэффициентов $c_0$ и $c_n$.


### 🧩 Задание 3.1 — МНК с базисными сплайнами

1. 🏭 **Базис.** Возьмите **кубические** $B$-сплайны (степень $3$):
   - внутренние узлы - равномерно по отрезку $[0,\, n-1]$
   - края - по $m$ **кратных** узлов: слева кратные $0$, справа кратные $n-1$ («зажатые» края);
   - значения $B_k(t)$ вычисляйте по **рекуррентной формуле Кокса - де Бора** (лекция 4); при кратных узлах знаменатели обнуляются — спасает «правило раскрытия неопределённостей» из лекции 4.
3. 🧮 **МНК.** Составьте матрицу фич $G_{ik} = B_k(t_i)$, правые части — матрицу $Y$ размера $(n,\, 80)$ и решите $\hat{\Theta}$ по нормальным уравнениям **одной операцией** (см. подсказки ниже).
4. 📈 **Восстановление.** Оцените координаты на плотной сетке шага $0.5$: $t = 0,\; 0.5,\; 1,\; \dots,\; n-1$ - и соберите массив формы $(2n-1,\, 40,\, 2)$, как в Части 1.
5. 🎬 **Визуализируйте** результат функцией `animate_lips` (interval=100).

**Результат** запишите в переменную `Task3LispPoints` — словарь, где ключ — буква,
а значение — массив $(2n-1,\, 40,\, 2)$.

---

#### ✅ Что должно получиться

- 🎬 Плавная анимация **без всплесков Рунге** на краях.
- 📉 На исходных кадрах кривая проходит **рядом, но не точно** — это сглаживание.
- 🎚️ При изменении $K$ плавность заметно меняется (проверим ниже).

In [ ]:
#TODO: Часть 3

Task3LispPoints = {}

def get_bspline_basis(t, k=3, num_basis=None):
    n = len(t)
    if num_basis is None:
        num_basis = max(k + 2, n // 2 + 1)

    # Количество внутренних узлов
    num_internal = num_basis - k - 1
    if num_internal < 1:
        num_internal = 1
        num_basis = k + 1 + num_internal
    
    # Внутренние узлы равномерно
    internal_knots = np.linspace(t[0], t[-1], num_internal + 2)[1:-1]
    
    # Вектор узлов: (k+1) кратных слева, внутренние, (k+1) кратных справа
    knots = [t[0]] * (k + 1) + list(internal_knots) + [t[-1]] * (k + 1)
    
    # Самопроверка
    assert len(knots) == num_basis + k + 1, \
        f"len(knots)={len(knots)}, expected {num_basis + k + 1}"
    
    # Строим матрицу G (n, num_basis)
    G = np.zeros((n, num_basis))
    for j in range(num_basis):
        coeffs = np.zeros(num_basis)
        coeffs[j] = 1.0
        basis_func = BSpline(knots, coeffs, k=k)
        G[:, j] = basis_func(t)
        
    return G, knots, k, num_basis

for letter, data in LipsData.items():
    n = data.shape[0]
    t = np.arange(n)
    t_new = np.arange(0, n, 0.5)
    
    Y = data.reshape(n, 80)
    
    # Матрица фич для исходных узлов
    G, knots, k, num_basis = get_bspline_basis(t, k=3)
    
    # Решаем нормальные уравнения: (G^T G) Theta = G^T Y
    GtG = G.T @ G
    GtY = G.T @ Y
    Theta = np.linalg.solve(GtG, GtY)
    
    # Матрица фич для новой плотной сетки
    G_new = np.zeros((len(t_new), num_basis))
    for j in range(num_basis):
        coeffs = np.zeros(num_basis)
        coeffs[j] = 1.0
        basis_func = BSpline(knots, coeffs, k=k)
        G_new[:, j] = basis_func(t_new)
        
    # Предсказание
    Y_new = G_new @ Theta
    new_data = Y_new.reshape(len(t_new), 40, 2)
    
    Task3LispPoints[letter] = new_data

# Визуализация
draw.animate_lips(Task3LispPoints['А'], interval=100)

> 💡 **Подсказки**
>
> - 🧱 **Сетка узлов.** Самопроверка базиса на любом $t$: каждая $B_k(t) \ge 0$, а сумма по всем базисным функциям тождественно равна единице: $\sum_k B_k(t) \equiv 1$.
> - 🧊 **Кратные узлы** ⇒ в рекурсии Кокса - де Бора появляются нули в знаменателях: именно для таких случаев в лекции 4 дано правило «вся дробь равна $0$».
> - 🍬 **Промышленная проверка.** Сравните свою матрицу фич с готовой реализацией:
>   ```python
>   from scipy.interpolate import BSpline
>   А потом посмотрите документацию, реализуйте через sklearn и сравните)))))
> - ✂️ **Одна матрица на всё.** Решать МНК по 80 координатам в цикле можно, только не пересчитывайте $G^{\mathsf{T}}G$ на каждой итерации 😄 Посчитайте один раз и решите сразу все системы:
>   ```python
>   Theta_hat = np.linalg.solve(G.T @ G, G.T @ Y)   # форма (K, 80)
>   ```

### 👀 На что смотреть в анимации

- 🎚️ **Прогоните $K = n/4$, $\;K = n/2$ и $K \approx n$.** При большом $K$ модель подкатывается к интерполяции и снова дёргается за данными; при маленьком — жирно сглаживает, но может «съесть» артикуляцию.
- 🧐 **Куда делись рывки?** МНК «размазывает» дёрганость кадров: губы движутся плавнее, но заострённые углы рта слегка округляются.

### 🌟 Эксперименты по желанию

- 📊 **Один график — три кривые.** Возьмите одну координату (например, $x(t)$ уголка рта — точка $0$) и нарисуйте на одном графике: точки исходных кадров плюс три кривые — Лагранж (Часть 1), интерполяционный кубический сплайн (Часть 2), МНК-сплайн (Часть 3).
- 🩺 **Числа обусловленности.** Посчитайте `np.linalg.cond(G.T @ G)` для сплайн-базиса и для степенного базиса $\{1,\, t,\, \dots,\, t^{K-1}\}$ той же мощности $K$ — и вспомните раздел «Почему степенной базис плох: матрица Гильберта» (лекция 5).
- 🔗 **МНК для слов (задание 2.2).** Прогоните тот же конвейер по склейкам звуков: сглаживание с умеренным $K$ дополнительно штопает стыки.

#### ✍️ Запишите здесь, что вы наблюдаете при изменении числа фич $K$,
####    и чем МНК-сглаживание отличается от интерполяции из предыдущих частей.
\#
    ВАШИ МЫСЛИ
\#

In [ ]:
# Наблюдения при изменении K:
# При малом K (сильное сглаживание) анимация очень плавная, но "съедает"
# резкие артикуляционные движения (губы не успевают сомкнуться).
# При большом K≈n МНК вырождается в обычную интерполяцию, и возвращаются
# мелкие дрожания и шум исходных кадров.
# Отличие от интерполяции: МНК не обязан проходить точно через исходные
# точки. Он находит "среднюю" траекторию, которая игнорирует случайные
# рывки детектора ключевых точек. Благодаря кратным узлам на краях, сплайн
# не "падает в ноль" на границах, а свободно аппроксимирует реальное положение губ,
#  что делает анимацию максимально естественной и лишенной артефактов Рунге.

## 🌟 Часть 4**** (необязательная). МНК + PCA

До сих пор мы обрабатывали **каждую из 80 координат независимо**.  
Но губы — это **связанная система**: точки двигаются согласованно, а не поодиночке.

**Метод главных компонент (PCA)** находит **опорные векторы** — направления, вдоль которых форма губ меняется сильнее всего.  
И тогда вместо 80 координат мы работаем с **несколькими коэффициентами** в новом базисе.

> 🎯 Интерполяция будет вестись уже **в пространстве главных компонент** —  
> по опорным векторам, которые описывают саму «суть» движения губ.  
> Это самый **естественный** и **устойчивый** подход из всех, что мы попробуем.

<div align="center">

# 🚧 ДАННАЯ ЗАДАЧА ОТКРЫВАЕТСЯ ПОСЛЕ СДАЧИ ОСНОВНОЙ ЧАСТИ И ВЫДАЁТСЯ ЛИЧНО ПРЕПОДАВАТЕЛЕМ!!!! 🚧

</div>